In [ ]:
"""
Supplementary Script: LLM-SAN Model Training and Ablation Study

This script provides a reproducible training pipeline for the LLM-SAN model 
on preprocessed dataset files (.pt format). It systematically performs 
ablation experiments by removing individual components (GCN, GRU, expert 
features) to evaluate their contribution to overall performance.

Key Features:
1. Deterministic Training:
   - Seeds are fixed across Python, NumPy, and PyTorch (CPU/GPU).
   - cuDNN is set to deterministic mode to ensure reproducibility.

2. Configurable Hyperparameters:
   - Batch size, hidden dimension, learning rate, training epochs, and random seed
     can be easily modified through the 'config' dictionary.

3. Model Evaluation:
   - The training and evaluation logic is encapsulated in the 'train_and_eval' function.

Usage Notes:
- Ensure that all dataset .pt files are placed in the 'Dataset' directory.
- The model implementation should be defined in 'model.py' and training logic
  in 'train.py'.
- The script is designed for GPU acceleration and requires PyTorch.

This script is intended for reproducibility and can be directly used as part of
the supplementary material for the ACL submission.
"""


import os
# ===============================
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import random
from model import LLMSAN
from train import train_and_eval

# ===============================
# ===============================
def set_seed(seed: int = 42):
    random.seed(seed)
    import numpy as np
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    torch.use_deterministic_algorithms(True)
    os.environ["PYTHONHASHSEED"] = str(seed)
# ===============================
config={"batch_size":1024,"hidden_dim":128,"learning_rate":1e-3,'seed':24,"epoch":3000}
set_seed(config["seed"])
model_type = "LLMSAN"
for pt_name in os.listdir("Dataset"):
    if pt_name.endswith(".pt"):
        config["ablation_type"]=model_type
        print(model_type)
        train_and_eval(pt_name, config)

In [ ]:
from performance import MAE_computation
# Training configuration dictionary
config = {
    "Dataset_with_label":"Dataset_with_label"
}
MAE_computation(config["Dataset_with_label"])